# Gerador de PDFs Premium (Medhelp)
Este notebook varre a pasta `Resumos_Prontos`, lê os arquivos Markdown gerados pelo Apps Script, realiza correções de listas e formatações, e gera PDFs com visualização premium (WeasyPrint).

**Recursos:**
- Correção automática de listas soltas e indentações de bullets
- Estilização de fluxos/ciclos (`A -> B -> C`) como chips conectados por setas
- Pula automaticamente resumos que já possuem PDF gerado

In [ ]:
!pip install -q weasyprint markdown
!apt-get install -y -q pango1.0-tools fonts-roboto

In [ ]:
import os
import re
import markdown
from weasyprint import HTML, CSS
from google.colab import drive

# Conectar Google Drive
drive.mount("/content/drive")

RESUMOS_DIR = "/content/drive/MyDrive/Logística - Drive/Transcrições/Resumos_Prontos"
PDFS_DIR = "/content/drive/MyDrive/Logística - Drive/Transcrições/PDFs_Premium"

os.makedirs(PDFS_DIR, exist_ok=True)

In [ ]:
CSS_PREMIUM = """
@page {
    size: A4;
    margin: 18mm 15mm 18mm 15mm;
    @bottom-center {
        content: counter(page);
        font-family: "Roboto", sans-serif;
        font-size: 9pt;
        color: #94a3b8;
    }
    @bottom-left {
        content: "© Conteúdo Autoral • João Gabriel R. Trovão";
        font-family: "Roboto", sans-serif;
        font-size: 8.5pt;
        color: #64748b;
    }
}

body {
    font-family: "Roboto", sans-serif;
    font-size: 10.5pt;
    line-height: 1.65;
    color: #1e293b;
    background-color: #ffffff;
}

/* ---- TÍTULOS E CABEÇALHOS PREMIUIM ---- */
h1, h2, h3, h4 {
    font-family: "Roboto", sans-serif;
    page-break-after: avoid;
}

h1 {
    font-size: 20pt;
    font-weight: 800;
    color: #0f172a;
    letter-spacing: -0.5px;
    padding-bottom: 8pt;
    margin-top: 0;
    margin-bottom: 1.2em;
    border-bottom: 3pt solid #0ea5e9;
}

h2 {
    font-size: 13.5pt;
    font-weight: 700;
    color: #0284c7;
    background-color: #f0f9ff;
    padding: 7pt 12pt;
    border-left: 4pt solid #0ea5e9;
    border-radius: 4pt;
    margin-top: 1.8em;
    margin-bottom: 0.8em;
}

h3 {
    font-size: 11.5pt;
    font-weight: 700;
    color: #0f172a;
    border-bottom: 1pt solid #e2e8f0;
    padding-bottom: 4pt;
    margin-top: 1.4em;
    margin-bottom: 0.5em;
}

strong {
    color: #0f172a;
    font-weight: 700;
}

blockquote {
    border-left: 3.5pt solid #0ea5e9;
    margin: 1.2em 0;
    padding: 10pt 14pt;
    background-color: #f8fafc;
    color: #0c4a6e;
    font-style: normal;
    border-radius: 0 6pt 6pt 0;
    page-break-inside: avoid;
}

/* ---- TABELAS ---- */
table {
    width: 100%;
    border-collapse: collapse;
    margin: 1.5em 0;
    font-size: 9.5pt;
    page-break-inside: auto;
    border-radius: 6pt;
    overflow: hidden;
}

tr {
    page-break-inside: avoid;
}

th {
    background-color: #f1f5f9;
    color: #0f172a;
    padding: 9pt 11pt;
    text-align: left;
    font-family: "Roboto", sans-serif;
    font-weight: 700;
    border-bottom: 2pt solid #cbd5e1;
}

td {
    padding: 8pt 11pt;
    border-bottom: 1pt solid #e2e8f0;
    background-color: #ffffff;
    vertical-align: top;
}

tbody tr:nth-child(even) td {
    background-color: #f8fafc;
}

/* ---- LISTAS E BULLETS NATIVOS ---- */
ul {
    padding-left: 18pt;
    margin: 0.8em 0;
    list-style-type: disc;
}

ul li::marker {
    color: #0ea5e9;
}

ul ul {
    padding-left: 16pt;
    margin: 4pt 0 4pt 0;
    list-style-type: circle;
}

ul ul li::marker {
    color: #64748b;
}

ol {
    padding-left: 18pt;
    margin: 0.8em 0;
}

li {
    margin-bottom: 5pt;
    line-height: 1.55;
}

/* ---- FLUXOS E CASCATAS (A -> B -> C) ---- */
.fluxo {
    display: flex;
    flex-wrap: wrap;
    align-items: center;
    gap: 6pt;
    margin: 12pt 0;
    padding: 9pt 12pt;
    background-color: #f0f9ff;
    border: 1pt solid #bae6fd;
    border-radius: 8pt;
    page-break-inside: avoid;
}

.passo {
    background-color: #ffffff;
    border: 1pt solid #7dd3fc;
    border-radius: 20pt;
    padding: 4pt 10pt;
    font-size: 9pt;
    color: #0369a1;
    font-weight: 600;
    box-shadow: 0 1px 2px rgba(0,0,0,0.04);
}

.seta {
    color: #0ea5e9;
    font-weight: 800;
    font-size: 11pt;
}
"""

def normalizar_indentacao_listas(md_texto):
    """Garante 4 espaços de recuo para sub-itens de lista."""
    padrao = re.compile(r'^( +)([-*+]|\d+\.)(\s+)', re.MULTILINE)
    def corrigir(m):
        espacos, marcador, resto = m.groups()
        novos_espacos = " " * (len(espacos) * 2) if len(espacos) < 4 else espacos
        return f'{novos_espacos}{marcador}{resto}'
    return padrao.sub(corrigir, md_texto)

def corrigir_listas_soltas(md_texto):
    """Insere linha em branco obrigatória antes de qualquer lista grudada no parágrafo."""
    linhas = md_texto.split('\n')
    padrao_item = re.compile(r'^\s*([-*+]|\d+\.)\s+')
    saida = []
    dentro_de_lista = False

    for linha in linhas:
        eh_item = bool(padrao_item.match(linha))
        linha_vazia = linha.strip() == ''

        if eh_item and not dentro_de_lista and saida and saida[-1].strip() != '':
            saida.append('')

        saida.append(linha)

        if eh_item:
            dentro_de_lista = True
        elif not linha_vazia:
            dentro_de_lista = False

    return '\n'.join(saida)

def estilizar_fluxos_html(html):
    """Transforma sequências de setas ('A -> B -> C') em chips visuais."""
    padrao_fluxo = re.compile(
        r'<p>((?:(?!</p>).)*?-&gt;(?:(?!</p>).)*?)</p>',
        re.DOTALL
    )

    def substituir(m):
        conteudo = m.group(1)
        passos = [p.strip() for p in conteudo.split('-&gt;')]
        if len(passos) < 2:
            return m.group(0)
        miolo = '<span class="seta">→</span>'.join(
            f'<span class="passo">{p}</span>' for p in passos
        )
        return f'<div class="fluxo">{miolo}</div>'

    return padrao_fluxo.sub(substituir, html)

processados = 0
ignorados = 0

for filename in os.listdir(RESUMOS_DIR):
    if filename.endswith(".md"):
        pdf_filename = filename.replace(".md", ".pdf")
        pdf_path = os.path.join(PDFS_DIR, pdf_filename)

        if os.path.exists(pdf_path):
            ignorados += 1
            continue

        filepath = os.path.join(RESUMOS_DIR, filename)

        with open(filepath, "r", encoding="utf-8") as f:
            md_content = f.read()

        # 1. Tratamento e Correção do Markdown
        md_content = normalizar_indentacao_listas(md_content)
        md_content = corrigir_listas_soltas(md_content)

        # 2. Conversão para HTML com extensões sane_lists, tables e fenced_code
        html_body = markdown.markdown(
            md_content,
            extensions=["tables", "fenced_code", "sane_lists"]
        )

        # 3. Pós-processamento visual de fluxos
        html_body = estilizar_fluxos_html(html_body)

        final_html = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="utf-8">
</head>
<body>
    {html_body}
</body>
</html>"""

        # 4. Renderização do PDF via WeasyPrint
        HTML(string=final_html).write_pdf(
            pdf_path,
            stylesheets=[CSS(string=CSS_PREMIUM)]
        )
        print(f"✅ PDF gerado com sucesso: {pdf_filename}")
        processados += 1

if processados == 0:
    print(f"Nenhum arquivo .md NOVO encontrado. ({ignorados} arquivos antigos já possuem PDF).")
else:
    print(f"\n🎉 Sucesso! {processados} PDFs gerados com qualidade premium. ({ignorados} antigos ignorados).")